# Notebook 01: Statistical & Time-Series Models (Family A)
**Objective:** Evaluate Naive, Seasonal Naive, Historical Mean, ETS, ARIMA, SARIMA, SARIMAX, and Prophet across 95 sparse series under Leave-One-Season-Year-Out (LOSO) and Forward-Chaining protocols.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

src_dir = Path.cwd().parent / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from data import load_model_data
from cv import get_loso_splits
from models_stat import TimeSeriesSeriesWrapper
from metrics import compute_core_metrics

df = load_model_data()
loso_splits = get_loso_splits(df)

# Evaluate ARIMA
wrapper = TimeSeriesSeriesWrapper(model_type="arima", min_obs=8)
test_fold = loso_splits[0]
tr_df = df.iloc[test_fold["train_idx"]]
te_df = df.iloc[test_fold["test_idx"]]
wrapper.fit(tr_df)
preds, cov = wrapper.predict(te_df)
m = compute_core_metrics(te_df["Total_Cost_Incl_Imputed"].values, preds, coverage=cov)
print(f"ARIMA Fold {test_fold['fold_name']}: MAE = Rs {m['MAE']:,.1f}, Coverage = {cov*100:.1f}%")


ARIMA Fold LOSO_2014: MAE = Rs 191,265.3, Coverage = 25.0%


### Summary of Family A Benchmark Results
Viewing the aggregated statistical family leaderboard from `results/leaderboard_loso.csv`.


In [2]:
results_csv = Path.cwd().parent / "results" / "leaderboard_loso.csv"
if results_csv.exists():
    df_lead = pd.read_csv(results_csv)
    stat_lead = df_lead[df_lead["family"] == "Statistical"]
    print(stat_lead.to_string(index=False))
else:
    print("Leaderboard will populate upon full pipeline completion.")


     family          model_name                                     run_id     LOSO_MAE    LOSO_RMSE       LOSO_R2    LOSO_MAPE  LOSO_sMAPE  LOSO_Log_RMSE  Coverage
Statistical PROPHET_REGULARIZED PROPHET_REGULARIZED_20261008_171816_c98601 5.562968e+04 2.325056e+05 -5.347000e-01 3.935000e+01       25.37         0.4375       1.0
Statistical             SARIMAX             SARIMAX_20261008_171245_3960f3 6.246348e+04 1.063232e+05  6.791000e-01 4.510000e+01       32.96         0.4507       1.0
Statistical     HISTORICAL_MEAN     HISTORICAL_MEAN_20261008_171054_ee7a9d 8.381394e+04 1.212613e+05  5.826000e-01 5.474000e+01       47.48         0.5270       1.0
Statistical      SEASONAL_NAIVE      SEASONAL_NAIVE_20261008_171053_f96400 1.135294e+05 1.782746e+05  9.780000e-02 9.651000e+01       55.52         0.7358       1.0
Statistical               NAIVE               NAIVE_20261008_170846_7ef99e 1.152197e+05 1.798923e+05  8.130000e-02 9.794000e+01       56.57         0.7485       1.0
Statistica